# FMG analysis runner for Google Colab

Open this notebook in Colab and run cells top-to-bottom. It clones the GitHub repository, mounts Drive, and runs the additive analysis scripts. Put your raw inputs in `MyDrive/FMG_data` (or change `DATA_DIR`).

Primary analyses use the exact package pins from the primary run manifest. Public-health analyses use a separate set of recorded pins. Select `RUN_ENVIRONMENT` before running the install cell. No manuscript or archived result file is edited. The S7 recreation will stop if it detects the known saved-metric mismatch.

In [ ]:
from pathlib import Path
import json, subprocess, sys, os, zipfile

REPO_URL = 'https://github.com/MVamsiKrishna2501050195/FMG-research.git'
REPO_ROOT = Path('/content/FMG-research')
if not (REPO_ROOT / '.git').exists():
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_ROOT)], check=True)
ANALYSIS = REPO_ROOT / 'analysis'
assert (ANALYSIS / 'fmg_decomposition.py').is_file(), 'Repository clone did not contain the analysis scripts'
PUBLIC_HEALTH_OUTPUTS = REPO_ROOT/'experiments/public_health/outputs'
if not (PUBLIC_HEALTH_OUTPUTS/'metrics_per_mask.csv').is_file():
    archive = REPO_ROOT/'experiments/public_health/outputs_full_2026_10_23/FMG_public_health_full_outputs.zip'
    if not archive.is_file():
        raise FileNotFoundError(f'Public-health baseline archive is missing: {archive}')
    PUBLIC_HEALTH_OUTPUTS.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive) as zf: zf.extractall(PUBLIC_HEALTH_OUTPUTS)
print('Repository:', REPO_ROOT)
print('Notebook Python:', sys.version)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DATA_DIR = Path('/content/drive/MyDrive/FMG_data')
DATA_DIR.mkdir(parents=True, exist_ok=True)
print('Put dataset files here:', DATA_DIR)

## Choose the analysis environment

Use `primary` for Adult/COMPAS/Taiwan decomposition and primary thresholds. Use `extension` for BRFSS/UCI public-health analyses. The two environments have different recorded package pins; do not combine them in one run. Changing this value after installation requires rerunning the pin-install cell.

In [ ]:
RUN_ENVIRONMENT = 'primary'  # choose 'primary' or 'extension'
assert RUN_ENVIRONMENT in {'primary', 'extension'}

if RUN_ENVIRONMENT == 'primary':
    primary_manifest = json.loads((REPO_ROOT/'run_manifest_20261025.json').read_text(encoding='utf-8'))
    PACKAGE_PINS = primary_manifest['environment']['packages']
else:
    extension_manifest_path = REPO_ROOT/'experiments/public_health/outputs/run_manifest.json'
    if extension_manifest_path.is_file():
        extension_manifest = json.loads(extension_manifest_path.read_text(encoding='utf-8'))
        PACKAGE_PINS = extension_manifest['software']['packages']
    else:
        requirements = REPO_ROOT/'experiments/public_health/requirements.txt'
        PACKAGE_PINS = {}
        for line in requirements.read_text(encoding='utf-8').splitlines():
            line = line.strip()
            if line and not line.startswith('#') and '==' in line:
                package, version = line.split('==', 1)
                PACKAGE_PINS[package] = version
print('Selected', RUN_ENVIRONMENT, 'environment pins:', PACKAGE_PINS)

## Install the selected pinned packages

This installs the versions recorded for the selected run. It does not edit the repository. If Colab asks for a runtime restart, restart it and rerun the notebook from the top.

In [ ]:
install_args = [sys.executable, '-m', 'pip', 'install', '--quiet']
install_args += [f'{package}=={version}' for package, version in PACKAGE_PINS.items()]
subprocess.run(install_args, check=True)
print('Pinned packages installed for:', RUN_ENVIRONMENT)

## Set or discover data paths

Place the original raw files in `MyDrive/FMG_data`; the cell looks for common filenames. If your filenames differ, edit the path variables manually.

In [ ]:
def first_file(*names):
    return next((str((DATA_DIR/name).resolve()) for name in names if (DATA_DIR/name).is_file()), None)

ADULT = first_file('adult.zip', 'adult.data')
COMPAS = first_file('compas-scores-two-years.csv')
TAIWAN = first_file('UCI_Credit_Card.csv', 'UCI_Credit_Card.xlsx.csv')
BRFSS = first_file('LLCP2024XPT.zip', 'brfss_2024.zip', 'LLCP2024.XPT')
UCI_DIABETES = first_file('diabetes+130-us-hospitals.zip', 'diabetic_data.csv')
RUN_GERMAN_OPENML = False

print({'Adult': ADULT, 'COMPAS': COMPAS, 'Taiwan': TAIWAN,
       'BRFSS': BRFSS, 'UCI Diabetes': UCI_DIABETES})
print('Missing inputs should remain None; no values will be guessed.')

In [ ]:
from importlib.metadata import version, PackageNotFoundError
installed = {}
for package in PACKAGE_PINS:
    try: installed[package] = version(package)
    except PackageNotFoundError: installed[package] = 'not-installed'
differences = {name:(expected, installed[name]) for name, expected in PACKAGE_PINS.items() if installed[name] != expected}
print('Package differences:', differences or 'none')
if differences:
    raise RuntimeError('Package versions differ from the selected run manifest; check install output or restart the Colab runtime.')

## Primary decomposition and primary thresholds

Run these only with `RUN_ENVIRONMENT='primary'` and all three primary data files. The scripts validate archived results and stop if checks fail.

In [ ]:
if RUN_ENVIRONMENT != 'primary':
    print('Skipped: select RUN_ENVIRONMENT = primary, install its pins, and rerun cells.')
elif not all([ADULT, COMPAS, TAIWAN]):
    print('Skipped: place adult.zip/adult.data, compas-scores-two-years.csv, and UCI_Credit_Card.csv in MyDrive/FMG_data.')
else:
    cmd = [sys.executable, str(ANALYSIS/'fmg_decomposition.py'), '--adult', ADULT, '--compas', COMPAS, '--taiwan', TAIWAN]
    if RUN_GERMAN_OPENML: cmd.append('--german-openml')
    subprocess.run(cmd, cwd=REPO_ROOT, check=True)

In [ ]:
if RUN_ENVIRONMENT == 'primary' and all([ADULT, COMPAS, TAIWAN]):
    cmd = [sys.executable, str(ANALYSIS/'threshold_sensitivity.py'), '--adult', ADULT, '--compas', COMPAS, '--taiwan', TAIWAN]
    if RUN_GERMAN_OPENML: cmd.append('--german-openml')
    subprocess.run(cmd, cwd=REPO_ROOT, check=True)
else:
    print('Skipped primary threshold run; select primary environment and provide the three primary datasets.')

## Public-health threshold and S7-style table

Run only after selecting `RUN_ENVIRONMENT='extension'` and installing its recorded pins. The S7 script will stop on the known mismatch rather than force a result. If it stops, share the full error output for diagnosis.

In [ ]:
if RUN_ENVIRONMENT != 'extension':
    print('Skipped: select extension environment, install its pins, then rerun this cell.')
elif not (BRFSS or UCI_DIABETES):
    print('Skipped: add the official BRFSS 2024 and/or UCI Diabetes raw file to MyDrive/FMG_data.')
else:
    cmd = [sys.executable, str(ANALYSIS/'threshold_sensitivity.py'), '--extension-only']
    if BRFSS: cmd += ['--brfss', BRFSS]
    if UCI_DIABETES: cmd += ['--diabetes-zip', UCI_DIABETES]
    subprocess.run(cmd, cwd=REPO_ROOT, check=True)
    cmd = [sys.executable, str(ANALYSIS/'extend_public_health_s7.py')]
    if BRFSS: cmd += ['--brfss', BRFSS]
    if UCI_DIABETES: cmd += ['--diabetes-zip', UCI_DIABETES]
    subprocess.run(cmd, cwd=REPO_ROOT, check=True)

## Tests and reporting helper

The primary environment runs analysis-package tests only. The extension environment also runs public-health primitive tests, which require its additional dependencies. The checklist runs only if an output CSV exists.

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', 'pytest'], check=True)
test_paths = [str(ANALYSIS/'tests')]
if RUN_ENVIRONMENT == 'extension':
    test_paths.append(str(REPO_ROOT/'experiments/public_health/test_audit_primitives.py'))
test_result = subprocess.run([sys.executable, '-m', 'pytest', '-q', *test_paths], cwd=REPO_ROOT)
if test_result.returncode:
    print('Tests failed; inspect the pytest output above. The analysis results are separate from these helper tests.')

for output_csv in [ANALYSIS/'outputs/fmg_decomposition_per_seed.csv', ANALYSIS/'outputs/public_health/public_health_s7_style_new.csv']:
    if output_csv.is_file():
        subprocess.run([sys.executable, str(ANALYSIS/'audit_checklist.py'), str(output_csv)], cwd=REPO_ROOT, check=True)

## Save results to Google Drive

Run this after the analysis cells. It copies the generated analysis outputs and a run log to Drive, outside the temporary Colab VM.

In [ ]:
from datetime import datetime
import shutil

RESULTS_DIR = DATA_DIR / f'fmg_results_{RUN_ENVIRONMENT}_{datetime.now().strftime("%Y%m%d_%H%M%S")}'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
for source in [ANALYSIS/'outputs', REPO_ROOT/'experiments/public_health/outputs']:
    if source.is_dir():
        destination = RESULTS_DIR / source.relative_to(REPO_ROOT)
        shutil.copytree(source, destination, dirs_exist_ok=True)
for source, name in [(REPO_ROOT/'run_manifest_20261025.json','primary_run_manifest.json'), (REPO_ROOT/'experiments/public_health/outputs/run_manifest.json','public_health_run_manifest.json')]:
    if source.is_file(): shutil.copy2(source, RESULTS_DIR/name)
print('Saved Colab outputs to:', RESULTS_DIR)